# 回测基本框架

QuantStudio 的一个主要功能就是对各种各样的模型使用历史数据进行回测，比如交易策略、绩效归因、因子选股能力等等。QuantStudio 提供了一个统一的框架来回测这些模型。回测功能集中在 `BackTest` 子模块中。

## 整体架构

回测模型逻辑上比较简单，就是沿着一个时间序列进行遍历，在每个时点进行一些计算，然后整理计算的结果形成回测报告。

QuantStudio 的回测框架建立在计算图引擎（`Core` 模块）之上，核心由四个组件构成：

- **`BTNode`** — 回测计算节点，继承自 `Node`，是各种回测算法的基类
- **`ReportNode`** — 报告生成节点基类，继承自 `BTNode`，具有 `ReportKey` 参数，负责将计算结果格式化为 HTML 报告
- **`BTReport`** — 回测报告汇总节点，继承自 `ReportNode`，将多个 `ReportNode` 的报告汇总成完整 HTML 报告
- **`BTStorer`** — 回测结果存储节点，继承自 `Node`，将回测结果持久化到 `BTResultDB`

回测框架按功能划分为四个子模块，外加一个结果持久化层：

| 组件 | 说明 | 核心类 |
|------|------|--------|
| `SectionFactor` | 截面因子测试 | `IC`, `ICReport`, `ICDecay`, `ICDecayReport`, `MultiPortfolio`, `FactorTurnover`, `SectionCorrelation`, `FamaMacBethRegression` |
| `Strategy` | 策略回测 | `MakeAccount`, `MakeStrategy`, `AccountReport` |
| `PerformanceAnalysis` | 绩效归因 | `BrinsonModel` |
| `Risk` | 风险模型测试 | `BiasTest` |
| 结果持久化 | 回测结果存储与管理 | `BTResultDB`, `HDF5BTResultDB`, `BTStorer` |

## 回测执行流程

回测遵循标准的 Node 生命周期（`init_compute` → `prepare_compute` → `compute`），由 `Engine` 驱动执行。在 `compute` 阶段，采用前向传播/反向传播模式：

1. **`init_compute`** — 初始化计算，处理时间范围（DTRange）
2. **`forward_compute`** — 前向传播，将时点信息传递给子节点
3. **`backward_compute`** — 反向传播，收集子节点的计算结果，执行回测分析逻辑

`BTReport` 通过读取各 `ReportNode` 依赖的 `ReportKey`，将报告汇总为完整 HTML。`BTStorer` 作为可选的下游节点，将结果持久化到磁盘。

## BTNode — 回测计算节点

`BTNode` 是回测框架的核心基类，所有回测分析类（如 `IC`、`AccountReport`、`BrinsonModel` 等）都继承自它。`BTNode` 负责计算逻辑，不涉及报告生成。

### 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"BTNode"` | 节点名称（冻结参数） |

### 核心方法

```python
class BTNode(Node):
    def init_compute(self, path, init_data, context) -> List[DTInitData]:
        """初始化计算。合并多个依赖节点的 DTRange，传递给子节点。"""

    def forward_compute(self, path, fwd_data, context) -> Tuple[List[DTLocalContext], DTLocalContext]:
        """前向传播。将时点列表传递给子节点。"""

    def backward_compute(self, path, bwd_data_list, context, local_context) -> dict:
        """反向传播。接收子节点计算结果，执行回测分析逻辑。子类重写此方法实现具体分析。"""

    def merge_result(self, result_list, context):
        """合并多个进程的计算结果（多进程模式下使用）。默认返回第一个结果。"""
```

### 生命周期详解

```
                   ┌───────────────┐
                   │ init_compute  │  ← 合并所有依赖节点的 DTRange
                   └──────┬────────┘
                          │
                   ┌──────▼────────┐
                   │forward_compute│  ← 将 DTs 传递给子节点
                   └──────┬────────┘
                          │
                   ┌──────▼─────────┐
                   │backward_compute│  ← 收集子节点结果，执行分析
                   └────────────────┘
```

`init_compute` 阶段通过 `context.NodeState` 记录合并后的 `dt_range`，后续子节点可访问此状态获取时点范围。如果节点在依赖链中出现循环引用（`self.QSID in path[:-1]`），返回空列表避免无限递归。

## ReportNode — 报告生成节点基类

`ReportNode` 继承自 `BTNode`，是所有报告生成节点的基类。它负责将计算节点的输出格式化为 HTML 报告。每个回测分析类型通常都有对应的 `ReportNode` 子类（如 `ICReport`、`AccountReport`、`BrinsonModelReport` 等）。

### 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"ReportNode"` | 节点名称（冻结参数） |
| `ReportKey` | `str` | `"Report"` | 在输出结果集中存储报告的键名（冻结参数） |

### 设计模式

`ReportNode` 子类以计算节点（`BTNode` 子类）为依赖，在 `backward_compute` 中接收计算结果，生成 HTML 报告并存入 `Output[ReportKey]`：

```python
class ICReport(ReportNode):
    def __init__(self, ic_node: IC, args={}, config_file=None, **kwargs):
        super().__init__(deps=[ic_node], args=args, config_file=config_file, **kwargs)

    def backward_compute(self, path, bwd_data_list, context, local_context=None):
        Output = bwd_data_list[0]          # 接收 IC 节点的计算结果
        HTML = self._genHTML(Output)        # 生成 HTML 报告
        Output[self._QSArgs.ReportKey] = HTML  # 存入结果集
        return Output
```

---

## BTReport — 回测报告汇总节点

`BTReport` 继承自 `ReportNode`，是报告容器节点，负责汇总多个 `ReportNode` 的报告为完整 HTML。

### 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"BTReport"` | 节点名称（冻结参数） |
| `ReportKey` | `str` | `"Report"` | 汇总报告的键名（冻结参数） |

### 构造方法

```python
class BTReport(ReportNode):
    def __init__(self, report_nodes: List[ReportNode] = [], args={}, config_file=None, **kwargs):
        """report_nodes: 产生报告的 ReportNode 列表"""
```

### 反向传播逻辑

`BTReport.backward_compute` 是报告汇总的核心：

1. 遍历所有依赖节点的 `backward_compute` 结果
2. 对每个依赖节点，读取其 `ReportKey` 参数，从结果中取出对应的 HTML 报告
3. 用分隔线将各模块的 HTML 拼接成完整报告
4. 最终返回包含汇总报告的结果字典

```python
for i, iOutput in enumerate(bwd_data_list):
    iReportKey = getattr(self.Deps[i]._QSArgs, "ReportKey", self._QSArgs.ReportKey)
    if iReportKey in iOutput:
        iHTML = iOutput[iReportKey]
    else:
        iHTML = "暂无报告"
    HTML += SepStr.format(Module=...) + iHTML
Output[self._QSArgs.ReportKey] = HTML
```

### 静态方法

```python
@staticmethod
def genOutputReport(output_list: List[dict], name_list: List[str] = None, report_key: str = "Report") -> str:
    """将多个输出字典列表合并为 HTML 报告字符串。适用于手动合并多个报告。"""
```

### 报告模板

`BTReport` 使用 HTML `<HR>` 分隔线和 `<div>` 模块标题来组织报告结构：

```html
<HR style="..."><div align="center"><strong>{模块序号}. {模块名称}</strong></div>
{模块HTML内容}
```

每个 `ReportNode` 子类的 `backward_compute` 方法负责生成规范化的 HTML 片段，通常包含：
- **参数设置**（无序列表）
- **统计表格**（带格式化的 pandas DataFrame HTML）
- **可视化图表**（base64 编码的 PNG 图片嵌入）

## 回测执行流程

QuantStudio 回测的核心特征：**因子/数据节点和回测节点是一起构造的**，它们通过依赖关系（`Deps`）串联成一张完整的计算图，然后在缓存、上下文、引擎三层环境中统一执行。计算图和引擎的详细机制参见 [计算图框架](../Core/计算图框架.ipynb)、[计算引擎](../Core/计算引擎.ipynb) 和 [缓存](../Core/缓存.ipynb)。

### 两个阶段：构造 vs 执行

整个回测分两阶段，边界在 `with Engine() as ExecEngine:` 处：

```python
# ===== 构造阶段：因子节点 + 回测节点一起构建 =====

Price = FT.getFactor("close")                             # 1. 叶子节点：从 HDF5DB 获取
FactorIC = CalcIC(...)(Factor, price=Price, mask=Mask)    # 2. 因子算子产生中间节点
ICNode = IC(FactorIC, args={"Name": "IC测试"})            # 3. 回测计算节点
ICReportNode = ICReport(ICNode)                           # 4. 报告生成节点
Report = BTReport(report_nodes=[ICReportNode])            # 5. 顶层报告汇总容器
Storer = BTStorer(deps=[Report], args={...})              # 6. (可选) 结果持久化节点

# ===== 执行阶段：Cache → Context → Engine =====

with FeatherFactorCache(...) as Cache:       # 最外层：缓存
    with FactorContext(...) as Context:      # 中间层：上下文（DTRuler/SectionIDs等）
        with Engine() as ExecEngine:         # 最内层：引擎调度 DAG
            Output, = ExecEngine.run([Storer], Context, ...)
```

### 构造阶段的依赖图

以一次 IC 回测为例，所有节点在构造阶段就通过 `Deps` 连接完毕：

```
BTStorer (可选, 持久化)
  └── BTReport (汇总报告)
        └── ICReport (ReportNode, 生成 IC 报告)
              └── IC (BTNode, 计算 IC 统计)
                    └── FactorIC (CalcIC 产生的因子节点)
                          ├── Price  ← HDF5DB
                          ├── Factor ← HDF5DB
                          └── Mask   ← HDF5DB
```

关键区别：**计算逻辑**（`IC`）和**报告生成**（`ICReport`）分离为两个节点，`BTReport` 只依赖报告节点，不直接依赖计算节点。

### 执行阶段：三层嵌套

| 层级 | 组件 | 职责 |
|------|------|------|
| 最外层 | `FeatherFactorCache` | 缓存计算结果到磁盘，避免重复计算 |
| 中间层 | `FactorContext` | 维护 DTRuler、SectionIDs、PID、缓存引用等全局状态 |
| 最内层 | `Engine` | 按 `init_compute → prepare_compute → compute` 生命周期驱动所有节点 |

### 最终输出

`run()` 返回一个列表，`BTReport` 的输出字典包含汇总报告（`ReportKey` 键）以及各依赖节点的中间结果：

```python
Output["Report"]        # 完整 HTML 报告字符串
Output["0-IC报告"]      # ICReport 节点的输出字典（含 Report 键和统计数据）
```

如果使用了 `BTStorer`，结果同时会被写入 `BTResultDB`，后续可通过 `db.readResult(group_name)` 重新加载，无需重新运行回测。

## 子模块

回测框架按功能分为四个子模块，每个子模块有独立的文档：

| 子模块 | 说明 | 详细文档 |
|--------|------|----------|
| `SectionFactor` | 截面因子测试：IC 分析、IC 衰减、分位数组合、因子换手率、截面相关性、Fama-MacBeth 回归 | [截面因子测试](截面因子测试.ipynb) |
| `Strategy` | 策略回测：账户创建、策略信号生成、净值与收益率计算、回测报告 | [策略回测](策略回测.ipynb) |
| `PerformanceAnalysis` | 绩效归因：Brinson 模型分解超额收益来源 | [业绩归因](业绩归因.ipynb) |
| `Risk` | 风险模型测试：Bias 检验验证风险预测准确性 | [风险模型测试](风险模型测试.ipynb) |
| 结果持久化 | 回测结果存储与管理：`BTResultDB` 接口、HDF5 实现、`BTStorer` 节点 | [BTStorer 设计文档](../../设计文档/BTStorer.md) |

## 回测结果持久化

`BTNode` / `BTReport` 的输出（嵌套 dict，叶节点为 DataFrame、Series、str、float 等）默认仅存在于内存中。通过 `BTStorer` 节点和 `BTResultDB` 结果库，可以将回测结果持久化到磁盘。

### 分层架构

回测结果持久化采用与因子框架相同的分层模式：

```
BTResultDB (抽象基类, 定义读写接口)
  └── HDF5BTResultDB   (目录模式 HDF5 实现, 每个结果组一个文件)

BTStorer (Node) — 插入计算图, 作为 BTNode/BTReport 的下游, 将结果写入 BTResultDB
```

### BTResultDB — 回测结果库

`BTResultDB` 定义了回测结果的读写接口：

| 方法 | 说明 |
|------|------|
| `writeResult(result, group_name, metadata=None)` | 写入一组回测结果 |
| `readResult(group_name)` | 按名称读取一组结果 |
| `listResults(metadata=None)` | 列出已存储的结果组, 支持按 metadata 筛选 |
| `readMetaData(group_name, key=None)` | 读取结果组的元信息 |
| `setMetaData(group_name, key=None, value=None, metadata=None)` | 设置结果组的元信息 |

`group_name` 支持路径层级（如 `"A股/IC/沪深300"`），`metadata` 是可选的标签 dict，用于后续查询筛选。

### BTStorer — 持久化节点

`BTStorer` 是一个 `Node`，插入计算图中作为 `BTReport` 的下游，将回测结果写入 `BTResultDB`。

**参数：**

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"BTStorer"` | 节点名称 |
| `TargetDB` | `BTResultDB` | 必填 | 目标结果库对象 |
| `GroupName` | `Optional[str]` | `None` | 结果组名称, 支持路径层级, `None` 时按依赖节点 Name 自动生成 |
| `Metadata` | `Optional[dict]` | `None` | 元信息标签, 透传给 `BTResultDB.writeResult` |

**两种接入方式：**

```python
# 方式 1: 存储整个 BTReport 的合并输出
Report = BTReport(report_nodes=[ICReportNode, AccountReportNode])
Storer = BTStorer(deps=[Report], args={"TargetDB": db, "GroupName": "A股/IC测试"})

# 方式 2: 存储单个 ReportNode 的输出
Storer = BTStorer(deps=[ICReportNode], args={"TargetDB": db, "Metadata": {"资产": "A股"}})
```

当 `BTStorer` 有多个依赖时，自动拆分为每个依赖一个子 Storer 实例（split 模式），解决并行 Engine 的写入冲突。

```
ReportNode1(ICReport) ─┐                          ├─→ BTStorer (GroupName="IC")         → ResultDB
ReportNode2(账户)  ────┼─→ BTStorer(拆分前) ──────┼─→ BTStorer (GroupName="账户报告")   → ResultDB
ReportNode3(相关)  ────┘                          └─→ BTStorer (GroupName="相关性")     → ResultDB
```

**读取结果：**

```python
result = db.readResult("A股/IC测试")
metadata = db.readMetaData("A股/IC测试")
all_groups = db.listResults(metadata={"资产": "A股"})
```

## 完整示例：IC 测试

下面展示一个使用回测框架进行 IC 测试的完整示例。

In [ ]:
import datetime as dt
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings('ignore', category=PerformanceWarning)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False
from IPython.display import HTML

from QuantStudio import __QS_MainPath__

In [2]:
# 参数设置
from QuantStudio.Factor.HDF5DB import HDF5DB
FDB = HDF5DB(args={"MainDir": Path(__QS_MainPath__).parent / "docs/data/HDF5"}).connect()

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 31)# 数据起止时间
TestStartDT, TestEndDT = dt.datetime(2025, 2, 28), EndDT# 测试起止时间

FT = FDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

# 再平衡时点序列
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime
BalanceDTs = getMonthLastDateTime(DTRuler)# 月末

In [ ]:
# 构建回测
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.BackTest.BackTestModel import BTReport, BTNode
from QuantStudio.BackTest.SectionFactor.IC import CalcIC, IC, ICReport

# 获取数据
FT = FDB.getTable("stock_cn_status")
Mask = (FT.getFactor("if_listed")==1)

FT = FDB.getTable("stock_cn_day_bar")
Price = FT.getFactor("close")

FT = FDB.getTable("stock_cn_factor_value")
FactorList = [FT.getFactor(iFactorName) for iFactorName in ["bp_lr", "ep_ttm"]]

# 步骤1: 创建因子算子 (计算 IC 因子)
FactorIC = CalcIC(
    descriptor_ids=SectionIDs,
    lookback=31,            # 在时间标尺上的回溯期数
    period_lookback=1,      # 在计算标尺上的回溯期数（月度IC, period_lookback=1表示用上月因子值）
    corr_method="spearman"  # 相关性方法: spearman / pearson / kendall
)(
    *FactorList,            # 待测试的因子列表
    price=Price,            # 价格因子（用于计算收益率）
    mask=Mask,              # 筛选条件（仅在上市状态为1时计算）
    factor_args={"CalcDTRuler": BalanceDTs}  # 按月计算 IC
)

# 步骤2: 创建回测计算节点
ICNode = IC(
    FactorIC,
    args={
        "Name": "IC测试",
        "RollingAvgPeriod": 2,  # IC 移动平均期数
    }
)

# 步骤3: 创建报告节点
ICReportNode = ICReport(ICNode)

# 步骤4: 创建报告汇总容器并执行
Report = BTReport(report_nodes=[ICReportNode])

CacheDir = Path(__QS_MainPath__).parent / "docs/data/Cache"
if not CacheDir.exists(): CacheDir.mkdir(parents=True)

with FeatherFactorCache(args={"DTRuler": DTRuler, "PIDs": ["0"], "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(PID="0", PIDList=["0"], DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Output, = ExecEngine.run(
                [Report],
                Context,
                fwd_data_list=[DTLocalContext(DTs=TestDTs)]
            )

# 步骤5: 查看报告
display(HTML(Output["Report"]))

## 关键设计模式

### 1. 算子-节点分离

回测框架采用与因子框架一致的"算子 + 节点"两层架构：

- **算子**（`PanelOperator` / `SectionOperator` 子类）：封装纯计算逻辑，通过 `calculate()` 方法实现，可复用
- **回测节点**（`BTNode` 子类）：封装依赖管理和生命周期方法

例如 IC 测试：`CalcIC`（算子）计算每个时点的 IC 值 → `IC`（回测节点）汇总统计数据。

### 2. 计算-报告分离

回测框架将**计算逻辑**和**报告生成**分离为两类节点：

- **`BTNode` 子类**（如 `IC`）：执行计算，输出结构化数据（DataFrame、Series 等）
- **`ReportNode` 子类**（如 `ICReport`）：依赖计算节点，将结构化数据格式化为 HTML 报告

这种分离的好处：
- 计算节点可以独立使用（如仅获取 IC 数据而不生成报告）
- 多个报告节点可以共享同一计算节点的结果
- `BTReport` 只汇总报告节点，职责清晰

### 3. 自定义回测节点

```python
# 步骤 1: 定义计算节点
class MyBackTest(BTNode):
    class __QS_ArgClass__(BTNode.__QS_ArgClass__):
        Name: str = Field(default="我的回测", frozen=True, title="名称")

    def __init__(self, factor, args={}, config_file=None, **kwargs):
        super().__init__(deps=[factor], args=args, config_file=config_file, **kwargs)

    def backward_compute(self, path, bwd_data_list, context, local_context=None):
        data = bwd_data_list[0]
        # 执行分析逻辑...
        return {"结果": data}

# 步骤 2: 定义报告节点
class MyBackTestReport(ReportNode):
    class __QS_ArgClass__(ReportNode.__QS_ArgClass__):
        Name: str = Field(default="我的回测报告", frozen=True, title="名称")

    def __init__(self, bt_node: MyBackTest, args={}, config_file=None, **kwargs):
        super().__init__([bt_node], args=args, config_file=config_file, **kwargs)

    def backward_compute(self, path, bwd_data_list, context, local_context=None):
        Output = bwd_data_list[0]
        html = "<h3>我的回测报告</h3>"
        html += Output["结果"].to_html()
        Output[self._QSArgs.ReportKey] = html
        return Output

# 步骤 3: 组装
bt = MyBackTest(some_factor)
report = MyBackTestReport(bt)
final_report = BTReport(report_nodes=[report])
```

## 完整示例：回测结果持久化

下面展示如何使用 `BTStorer` 将上面的 IC 测试结果持久化到磁盘，并在后续读取使用。

In [ ]:
# 构建带持久化的回测: BTReport → BTStorer
from QuantStudio.BackTest.BTResultDB import HDF5BTResultDB
from QuantStudio.BackTest.BTStorer import BTStorer

# 创建结果库 (目录模式)
ResultDir = Path(__QS_MainPath__).parent / "docs/data/BTResults"
ResultDB = HDF5BTResultDB(args={"MainDir": str(ResultDir)})

# 复用上面的 ICNode 和 ICReportNode, 在 BTReport 之后接入 BTStorer
ICReportNode2 = ICReport(ICNode)
Report2 = BTReport(report_nodes=[ICReportNode2])
Storer = BTStorer(
    deps=[Report2],
    args={
        "TargetDB": ResultDB,
        "GroupName": "A股/IC测试",
        "Metadata": {"资产": "A股", "策略": "IC", "日期范围": f"{TestStartDT:%Y%m%d}-{TestEndDT:%Y%m%d}"},
    }
)

with FeatherFactorCache(args={"DTRuler": DTRuler, "PIDs": ["0"], "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(PID="0", PIDList=["0"], DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Output2, = ExecEngine.run([Storer], Context, fwd_data_list=[DTLocalContext(DTs=TestDTs)])

print("写入完成")

In [ ]:
# 从结果库读取回测结果
ResultDB = HDF5BTResultDB(args={"MainDir": str(ResultDir)})

# 列出所有结果组
print("所有结果组:", ResultDB.ResultNames)

# 按 metadata 筛选
a_stock_results = ResultDB.listResults(metadata={"资产": "A股"})
print("A股结果:", a_stock_results)

# 读取结果
loaded = ResultDB.readResult("A股/IC测试")
print("\n结果 keys:", list(loaded.keys()))
print("\n统计数据:")
display(loaded["统计数据"])